<a href="https://colab.research.google.com/github/sanchi23002/COMPUTER_VISION_WITH_OPENCV_AND_DEEP_LEARNING/blob/main/NN_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from numpy import genfromtxt

In [2]:
data = genfromtxt('bank_note_data.txt',delimiter= ',')

In [3]:
data

array([[  3.6216 ,   8.6661 ,  -2.8073 ,  -0.44699,   0.     ],
       [  4.5459 ,   8.1674 ,  -2.4586 ,  -1.4621 ,   0.     ],
       [  3.866  ,  -2.6383 ,   1.9242 ,   0.10645,   0.     ],
       ...,
       [ -3.7503 , -13.4586 ,  17.5932 ,  -2.7771 ,   1.     ],
       [ -3.5637 ,  -8.3827 ,  12.393  ,  -1.2823 ,   1.     ],
       [ -2.5419 ,  -0.65804,   2.6842 ,   1.1952 ,   1.     ]])

In [4]:
labels = data[:,4]
labels

array([0., 0., 0., ..., 1., 1., 1.])

In [6]:
features = data[:,0:4]
features

array([[  3.6216 ,   8.6661 ,  -2.8073 ,  -0.44699],
       [  4.5459 ,   8.1674 ,  -2.4586 ,  -1.4621 ],
       [  3.866  ,  -2.6383 ,   1.9242 ,   0.10645],
       ...,
       [ -3.7503 , -13.4586 ,  17.5932 ,  -2.7771 ],
       [ -3.5637 ,  -8.3827 ,  12.393  ,  -1.2823 ],
       [ -2.5419 ,  -0.65804,   2.6842 ,   1.1952 ]])

In [7]:
X = features
Y = labels

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size = 0.33, random_state = 42)# random_states is always 42

The random_state parameter in train_test_split (and many other functions in scikit-learn that involve randomness) is used to control the shuffling applied to the data before splitting.

Here's what it does:

Reproducibility: When you set random_state to an integer (like 42), the random number generator used for shuffling will produce the same sequence of random numbers every time you run the code. This ensures that the split of your data into training and testing sets will always be identical. This is crucial for reproducibility, as it means anyone running your code will get the same results.

In [9]:
print(len(X_train))
print(len(X))
print(len(y_train))
print(len(Y))

919
1372
919
1372


In [10]:
#we will normalize the data but always test and training set differently
from sklearn.preprocessing import MinMaxScaler
## creating a object that will scale all data
scaler_object = MinMaxScaler()
scaler_object.fit(X_train)
scaled_X_train = scaler_object.transform(X_train)
scaled_X_test = scaler_object.transform(X_test)

When using scikit-learn's MinMaxScaler:

.fit() calculates the parameters needed for the transformation. Specifically, it finds the minimum ($X_{min}$) and maximum ($X_{max}$) values in the dataset you pass to it.

.transform() applies the normalization formula using those pre-calculated parameters:$$X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}$$

✅scaler_object.transform(X_test) $\rightarrow$ Scale the test data using the training data's scales

✅Scenario A: You fit the scaler on the test data separately

The Problem: The exact same numerical value will be mapped to entirely different scaled values in your training and testing sets.

The Result: Imagine your model learned during training that an input of 1.0 means "Very High Risk". If your test set has a lower overall maximum, a moderate value in your test set might suddenly get scaled to 1.0. Your model will misinterpret this moderate value as "Very High Risk" and make an incorrect prediction.

✅Scenario B: You don't scale the test data at all

The Problem: Your Keras Sequential model (which you are importing at the bottom of your screenshot) uses weights and biases that were fine-tuned specifically for inputs between 0 and 1.

The Result: If your raw feature is something like "Income" with a value of 50,000, and you pass that 50,000 directly into a neuron expecting a maximum of 1.0, the math completely explodes. The activation functions (like Sigmoid or ReLU) will saturate, the outputs will turn into nonsense, and your model's accuracy will plummet to near-zero.

In [12]:
from keras.models import Sequential
from keras.layers import Dense

In [34]:
model = Sequential()
## in only input layer you need to pass input dimention(dense layer means all data connected to each other)
model.add(Dense(4, input_dim = 4, activation = 'relu'))
# this is hidden later
model.add(Dense(8, activation = 'relu'))

model.add(Dense(16, activation = 'relu'))
# this is output layer
model.add(Dense(1, activation = 'sigmoid'))

model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy'])



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


While the loss function is what the computer uses to optimize the math, it's often hard for human beings to read a cross-entropy score of 0.234 and instantly know how well the model is performing.

Why Accuracy? By passing metrics=['accuracy'], you tell Keras to calculate and print out the percentage of correctly guessed samples during training and validation (e.g., 0.85 means 85% correct).

Unlike the loss function, metrics are purely for your eyes and do not affect the mathematical updates to the network weights.

In [35]:
model.fit(scaled_X_train, y_train, epochs = 50, verbose = 2 )

Epoch 1/50
29/29 - 2s - 52ms/step - accuracy: 0.5550 - loss: 0.6621
Epoch 2/50
29/29 - 0s - 4ms/step - accuracy: 0.5669 - loss: 0.6455
Epoch 3/50
29/29 - 0s - 4ms/step - accuracy: 0.6083 - loss: 0.6278
Epoch 4/50
29/29 - 0s - 3ms/step - accuracy: 0.6453 - loss: 0.6098
Epoch 5/50
29/29 - 0s - 4ms/step - accuracy: 0.6823 - loss: 0.5883
Epoch 6/50
29/29 - 0s - 4ms/step - accuracy: 0.6931 - loss: 0.5608
Epoch 7/50
29/29 - 0s - 3ms/step - accuracy: 0.7421 - loss: 0.5195
Epoch 8/50
29/29 - 0s - 4ms/step - accuracy: 0.7780 - loss: 0.4713
Epoch 9/50
29/29 - 0s - 4ms/step - accuracy: 0.8085 - loss: 0.4299
Epoch 10/50
29/29 - 0s - 4ms/step - accuracy: 0.8248 - loss: 0.3940
Epoch 11/50
29/29 - 0s - 4ms/step - accuracy: 0.8215 - loss: 0.3647
Epoch 12/50
29/29 - 0s - 4ms/step - accuracy: 0.8400 - loss: 0.3394
Epoch 13/50
29/29 - 0s - 4ms/step - accuracy: 0.8553 - loss: 0.3166
Epoch 14/50
29/29 - 0s - 3ms/step - accuracy: 0.8749 - loss: 0.2885
Epoch 15/50
29/29 - 0s - 3ms/step - accuracy: 0.8955 - l

In model.fit(), verbose controls how much information (logs and text) Keras prints on your screen while the model is training.

Think of it as a "logging level" or a volume knob for the model's communication. It doesn't change how your model trains, only how much it talks to you while doing it.

Keras provides four main settings for verbose: 0, 1, 2, and 3.

⚡1. verbose = 0 (Silent Mode)
What it means: Completely silent.

What you see: Absolutely nothing appears on the screen while the model trains.

When to use it: When you are running your code in production, inside a background script, or executing thousands of automated hyperparameter trials where printing text would clutter your log files.

⚡verbose = 1 (Progress Bar Mode — The Default)
What it means: Highly detailed with an animated progress bar for every single epoch.

What you see: ```text
Epoch 1/50
250/250 [==============================] - 2s 4ms/step - loss: 0.6543 - accuracy: 0.7120

When to use it: Ideal when running a model interactively in a Jupyter Notebook or VS Code for a few epochs, as it gives you a real-time visual cue of how fast each epoch is processing.

⚡verbose = 2 (One Line Per Epoch Mode — Your Setting)
What it means: It removes the animated progress bar and prints exactly one clean line of text after each epoch finishes.

What you see:

Plaintext


Epoch 1/50 - 2s - loss: 0.6543 - accuracy: 0.7120


Epoch 2/50 - 1s - loss: 0.5122 - accuracy: 0.7850


Epoch 3/50 - 1s - loss: 0.4311 - accuracy: 0.8210


Why this is a great choice: Since you are training for 50 epochs, using verbose=1 would generate a massive, constantly moving wall of text that stretches your notebook completely out of bounds. verbose=2 gives you a tidy, 50-line summary that lets you easily scroll back and track exactly how your loss and accuracy changed over time.

⚡verbose = 3 (Minimal Mode)
What it means: It only prints the epoch number, completely omitting the progress bar and the final metrics.

What you see:

Plaintext


Epoch 1/50


Epoch 2/50


When to use it: Rare, but helpful if you just want to know that your training hasn't frozen, without caring about the intermediate scores until the very end.

In [36]:
model.predict(scaled_X_test)

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


array([[2.0492221e-03],
       [4.7616279e-01],
       [1.5476851e-01],
       [2.5676377e-04],
       [3.3709493e-03],
       [5.6264939e-04],
       [4.8351013e-03],
       [6.9775298e-04],
       [9.7341998e-04],
       [1.5033209e-03],
       [8.8177115e-01],
       [9.9583888e-01],
       [6.3547667e-04],
       [9.9476093e-01],
       [2.9981660e-03],
       [8.1621647e-01],
       [9.8974156e-01],
       [9.8954785e-01],
       [9.9965864e-01],
       [9.1475409e-01],
       [1.4750297e-03],
       [6.6882424e-04],
       [9.8313123e-01],
       [1.1113239e-03],
       [9.9911886e-01],
       [1.2570440e-03],
       [3.5452741e-04],
       [9.9735940e-01],
       [8.0948252e-05],
       [2.2064739e-04],
       [9.9292797e-01],
       [1.3663252e-03],
       [2.0246925e-03],
       [9.9849778e-01],
       [9.9834627e-01],
       [5.5254746e-04],
       [8.5307789e-01],
       [9.7214472e-01],
       [9.9632365e-01],
       [1.1697119e-03],
       [3.1871279e-04],
       [9.894498

In [42]:
predictions = (model.predict(scaled_X_test>0.5)).astype("int32")
#predictions

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step


In [39]:
model.metrics_names

['loss', 'compile_metrics']

In [26]:
from sklearn.metrics import confusion_matrix, classification_report

💯. confusion_matrix (The Raw Count)

A Confusion Matrix is a $2 \times 2$ grid (for binary classification) that cross-references the Actual True Labels against the Predicted Labels generated by your model. It counts exactly how many times the model got things right or wrong.

The Structure

When you run it, it returns a 2D NumPy array structured like this:

Predicted 0       Predicted 1


Actual 0 [ [   True Negatives,   False Positives ],


Actual 1   [   False Negatives,  True Positives  ] ]

✅
✅2. classification_report (The Analysis Report)


While the confusion matrix gives you raw numbers, classification_report takes those numbers and automatically calculates four vital mathematical metrics for each individual class (0 and 1).

A. Precision (Quality of positive predictions)Formula: $\frac{TP}{TP + FP}$

B. Recall / Sensitivity (Quantity of actual positives found)Formula: $\frac{TP}{TP + FN}$

C. F1-Score (The Balance)Formula: $2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$

In [44]:
predictions = (model.predict(scaled_X_test>0.5)).astype("int32")
#predictions

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


In [45]:
confusion_matrix(y_test, predictions)

array([[257,   0],
       [163,  33]])

In [46]:
print(classification_report(y_test,predictions))

              precision    recall  f1-score   support

         0.0       0.61      1.00      0.76       257
         1.0       1.00      0.17      0.29       196

    accuracy                           0.64       453
   macro avg       0.81      0.58      0.52       453
weighted avg       0.78      0.64      0.56       453



In [48]:
model.save('my_model.keras')

In [49]:
from keras.models import load_model

In [50]:
newmodel = load_model('my_model.keras')

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [52]:
newmodel.predict(scaled_X_test)

15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step


array([[2.0492221e-03],
       [4.7616279e-01],
       [1.5476851e-01],
       [2.5676377e-04],
       [3.3709493e-03],
       [5.6264939e-04],
       [4.8351013e-03],
       [6.9775298e-04],
       [9.7341998e-04],
       [1.5033209e-03],
       [8.8177115e-01],
       [9.9583888e-01],
       [6.3547667e-04],
       [9.9476093e-01],
       [2.9981660e-03],
       [8.1621647e-01],
       [9.8974156e-01],
       [9.8954785e-01],
       [9.9965864e-01],
       [9.1475409e-01],
       [1.4750297e-03],
       [6.6882424e-04],
       [9.8313123e-01],
       [1.1113239e-03],
       [9.9911886e-01],
       [1.2570440e-03],
       [3.5452741e-04],
       [9.9735940e-01],
       [8.0948252e-05],
       [2.2064739e-04],
       [9.9292797e-01],
       [1.3663252e-03],
       [2.0246925e-03],
       [9.9849778e-01],
       [9.9834627e-01],
       [5.5254746e-04],
       [8.5307789e-01],
       [9.7214472e-01],
       [9.9632365e-01],
       [1.1697119e-03],
       [3.1871279e-04],
       [9.894498